# <font color="#418FDE" size="6.5" uppercase>**PCA SVD**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Apply PCA and interpret explained variance and components. 
- Use incremental, sparse, kernel, and mini-batch PCA variants appropriately. 
- Apply TruncatedSVD for sparse high-dimensional data such as text features. 


## **1. PCA Basics**

### **1.1. Principal Components**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_A/image_01_01.jpg?v=1784025077" width="250">



>* New axes capture greatest data variation
>* Correlated features become simpler uncorrelated summaries

>* Loadings show each variable’s component contribution
>* Components reveal patterns, not new information

>* Summarize overlapping high-dimensional patterns
>* Interpret components with context and caution



In [ ]:
#@title Python Code - Principal Components

# This example shows principal component directions.
# PCA rotates correlated features into new axes.
# The plot reveals variance captured by components.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import sklearn

# Create a small correlated dataset with fixed randomness.
rng = np.random.default_rng(42)
size = rng.normal(0, 1, 80)

# Build two features that mostly move together.
noise = rng.normal(0, 0.35, 80)
area_score = size + noise
price_score = 1.8 * size + rng.normal(0, 0.45, 80)

# Combine the two original features into one matrix.
features = np.column_stack((area_score, price_score))
if features.shape != (80, 2):
    raise ValueError("Expected 80 rows and 2 features.")

# Standardize features before PCA compares their variation.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Fit PCA and transform data into principal component scores.
pca = PCA(n_components=2)
component_scores = pca.fit_transform(scaled_features)

# Print concise values that explain the component directions.
print("scikit-learn version:", sklearn.__version__)
print("Explained variance ratios:", np.round(pca.explained_variance_ratio_, 3))
print("PC1 weights for area and price:", np.round(pca.components_[0], 3))
print("PC2 weights for area and price:", np.round(pca.components_[1], 3))

# Plot the data after rotation into principal component coordinates.
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(component_scores[:, 0], component_scores[:, 1], alpha=0.75)
ax.axhline(0, color="gray", linewidth=1)
ax.axvline(0, color="gray", linewidth=1)
ax.set_title("Data Rotated Into Principal Components")
ax.set_xlabel("Principal Component 1 score")
ax.set_ylabel("Principal Component 2 score")
plt.show()



### **1.2. Centered SVD**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_A/image_01_02.jpg?v=1784025075" width="250">



>* Center features to measure deviations from averages
>* SVD finds main directions of variation

>* SVD finds components, variation, and scores
>* Components reveal shared patterns across features

>* Scale features to prevent domination
>* Components summarize variation around the mean



### **1.3. Explained Variance**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_A/image_01_03.jpg?v=1784025079" width="250">



>* Variance shows each component’s captured information
>* Fewer dimensions can preserve key structure

>* Variance ratios guide component selection
>* Spread-out variance warns against over-reduction

>* Judge variance with domain context
>* Standardize data and inspect component loadings



In [ ]:
#@title Python Code - Explained Variance

# This example shows PCA explained variance.
# Components summarize standardized wine measurements.
# The plot reveals retained information.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Load a small packaged dataset with numeric features.
wine = load_wine()
features = wine.data

# Validate the feature matrix before applying PCA.
if features.ndim != 2 or features.shape[1] < 2:
    raise ValueError("PCA needs a two-dimensional feature matrix.")

# Standardize features so units do not dominate variance.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Fit PCA once, keeping all possible components.
pca = PCA(random_state=42)
pca.fit(scaled_features)

# Convert ratios to percentages for beginner-friendly reading.
variance_percent = pca.explained_variance_ratio_ * 100
cumulative_percent = np.cumsum(variance_percent)

print("scikit-learn version:", sklearn.__version__)
print("First component explains:", round(variance_percent[0], 1), "%")
print("First two components explain:", round(cumulative_percent[1], 1), "%")
print("Components needed for at least 90%:", int(np.argmax(cumulative_percent >= 90) + 1))

# Plot cumulative explained variance across components.
fig, ax = plt.subplots(figsize=(7, 4))
component_numbers = np.arange(1, len(cumulative_percent) + 1)
ax.plot(component_numbers, cumulative_percent, marker="o", label="Cumulative variance")
ax.axhline(90, color="gray", linestyle="--", label="90% target")
ax.set_title("PCA Explained Variance on Wine Data")
ax.set_xlabel("Number of principal components")
ax.set_ylabel("Cumulative explained variance (%)")
ax.set_ylim(0, 105)
ax.legend()
plt.show()



## **2. PCA Variants**

### **2.1. Whitening Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_A/image_02_01.jpg?v=1784025066" width="250">



>* Whitening equalizes PCA component scales
>* Prevents high-variance components dominating models

>* Balances PCA features for downstream models
>* Gives subtle components equal influence

>* Whitening can amplify low-variance noise
>* Use for scaled features, not interpretation



In [ ]:
#@title Python Code - Whitening Features

# This example compares PCA with whitening.
# Whitening equalizes retained component variances.
# The printed values show the scale change.

import numpy as np
import sklearn
from sklearn.decomposition import PCA
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler

# Create correlated numeric features for a small lesson.
features, labels = make_classification(
    n_samples=300,
    n_features=5,
    n_informative=3,
    n_redundant=2,
    random_state=42,
)

# Standardize first so PCA focuses on correlation structure.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Validate the expected two-dimensional input shape.
if scaled_features.ndim != 2 or scaled_features.shape[1] != 5:
    raise ValueError("Expected a table with five numeric features.")

# Fit ordinary PCA and whitening PCA with matching components.
plain_pca = PCA(n_components=3, random_state=42)
white_pca = PCA(n_components=3, whiten=True, random_state=42)

plain_scores = plain_pca.fit_transform(scaled_features)
white_scores = white_pca.fit_transform(scaled_features)

# Compare component variances after each transformation.
plain_variance = np.var(plain_scores, axis=0, ddof=1)
white_variance = np.var(white_scores, axis=0, ddof=1)

print("scikit-learn version:", sklearn.__version__)
print("Explained variance ratio:", np.round(plain_pca.explained_variance_ratio_, 3))
print("Ordinary PCA score variances:", np.round(plain_variance, 3))
print("Whitened PCA score variances:", np.round(white_variance, 3))
print("Whitening keeps components uncorrelated and rescales them equally.")



### **2.2. Incremental PCA**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_A/image_02_02.jpg?v=1784025064" width="250">



>* Handles large or streaming datasets in batches
>* Updates components without storing all data

>* Batch size balances memory and stability
>* Scalable PCA keeps components interpretable

>* Best for large or streaming datasets
>* Preprocess features before interpreting components



In [ ]:
#@title Python Code - Incremental PCA

# Demonstrate Incremental PCA on manageable data batches.
# Compare batch learning with ordinary PCA results.
# Show explained variance and two-dimensional projections.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.decomposition import IncrementalPCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Create a small deterministic dataset with correlated numeric features.
features, labels = make_classification(
    n_samples=1200,
    n_features=12,
    n_informative=6,
    n_redundant=4,
    random_state=42,
)

# Validate the shape before fitting decomposition models.
if features.shape != (1200, 12):
    raise ValueError("Expected 1200 rows and 12 features.")

# Scale features so PCA is not dominated by large units.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Fit Incremental PCA by feeding the data in small batches.
incremental_pca = IncrementalPCA(n_components=2, batch_size=150)
for start in range(0, scaled_features.shape[0], 150):
    batch = scaled_features[start:start + 150]
    incremental_pca.partial_fit(batch)

# Transform all rows after the incremental updates are complete.
incremental_projection = incremental_pca.transform(scaled_features)

# Fit ordinary PCA once for a simple reference comparison.
ordinary_pca = PCA(n_components=2, random_state=42)
ordinary_projection = ordinary_pca.fit_transform(scaled_features)

# Print concise results that connect batches to explained variance.
print("scikit-learn version:", sklearn.__version__)
print("Rows processed per batch:", incremental_pca.batch_size)
print("Incremental PCA variance:", np.round(incremental_pca.explained_variance_ratio_, 3))
print("Ordinary PCA variance:", np.round(ordinary_pca.explained_variance_ratio_, 3))

# Plot the incremental two-component representation.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    incremental_projection[:, 0],
    incremental_projection[:, 1],
    c=labels,
    cmap="viridis",
    s=18,
    alpha=0.75,
)

# Label the plot so the reduced dimensions are easy to read.
ax.set_title("Incremental PCA projection from batch updates")
ax.set_xlabel("Incremental principal component 1")
ax.set_ylabel("Incremental principal component 2")
ax.legend(*scatter.legend_elements(), title="Class")
plt.show()



### **2.3. Sparse PCA**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_A/image_02_03.jpg?v=1784025071" width="250">



>* Sparse PCA uses fewer features per component
>* Improves interpretability while reducing dimensionality

>* Trades some variance for clearer components
>* Use when interpretability matters most

>* Center, scale, and tune sparsity carefully
>* Seek stable, interpretable feature groups



In [ ]:
#@title Python Code - Sparse PCA

# Sparse PCA makes components easier to interpret.
# This example compares dense and sparse loadings.
# The output highlights selected original features.

import numpy as np
import sklearn
from sklearn.decomposition import PCA
from sklearn.decomposition import SparsePCA
from sklearn.preprocessing import StandardScaler

# Create small data with three hidden feature groups.
rng = np.random.default_rng(42)
samples = 180
hidden = rng.normal(size=(samples, 3))

# Each observed feature mostly follows one hidden group.
weights = np.array(
    [[1.0, 0.9, 0.8, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.9, 0.8], [0.6, 0.0, 0.0, 0.0, 0.0, 0.7]]
)
noise = rng.normal(scale=0.15, size=(samples, 6))
X = hidden @ weights + noise

# Scale features before decomposition.
feature_names = np.array(["survey_A", "survey_B", "survey_C", "spend_D", "spend_E", "spend_F"])
X_scaled = StandardScaler().fit_transform(X)

# Validate the teaching dataset shape.
if X_scaled.shape != (samples, 6):
    raise ValueError("Expected 180 rows and 6 features.")

# Fit ordinary PCA and Sparse PCA with two components.
pca = PCA(n_components=2, random_state=42)
sparse_pca = SparsePCA(n_components=2, alpha=1.2, random_state=42, max_iter=1000)
pca.fit(X_scaled)
sparse_pca.fit(X_scaled)

# Count meaningful feature weights in each component.
threshold = 0.20
pca_counts = np.sum(np.abs(pca.components_) > threshold, axis=1)
sparse_counts = np.sum(np.abs(sparse_pca.components_) > threshold, axis=1)

# Show how sparsity changes interpretability.
print("scikit-learn version:", sklearn.__version__)
print("PCA nonzero-like loadings per component:", pca_counts.tolist())
print("Sparse PCA nonzero-like loadings per component:", sparse_counts.tolist())

# Display the strongest Sparse PCA features.
for component_index in range(2):
    loadings = sparse_pca.components_[component_index]
    selected = feature_names[np.abs(loadings) > threshold]
    rounded = np.round(loadings[np.abs(loadings) > threshold], 2)
    print("Sparse component", component_index + 1, "uses", selected.tolist(), rounded.tolist())



## **3. Truncated SVD**

### **3.1. Mini Batch Sparse PCA**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_A/image_03_01.jpg?v=1784025073" width="250">



>* Sparse components highlight fewer important features
>* Mini batches scale PCA to huge data

>* Reduces large sparse data in manageable batches
>* Creates interpretable components from focused feature groups

>* TruncatedSVD suits sparse text embeddings
>* Mini batch sparse PCA improves interpretability



In [ ]:
#@title Python Code - Mini Batch Sparse PCA

# This example demonstrates mini batch sparse PCA.
# Sparse components highlight small feature groups.
# The output compares compact transformed features.

import numpy as np
import sklearn
from sklearn.decomposition import MiniBatchSparsePCA
from sklearn.feature_extraction.text import CountVectorizer

# Short documents create a tiny sparse text matrix.
documents = [
    "exam grade assignment rubric feedback",
    "quiz exam grade study feedback",
    "tuition scholarship financial aid payment",
    "financial aid tuition billing payment",
    "login password portal access error",
    "portal login technical error access",
]

# CountVectorizer builds word-count features from text.
vectorizer = CountVectorizer()
sparse_counts = vectorizer.fit_transform(documents)
feature_names = vectorizer.get_feature_names_out()

# MiniBatchSparsePCA needs a dense array for this small demo.
word_counts = sparse_counts.toarray().astype(float)
if word_counts.shape[0] != len(documents):
    raise ValueError("The document matrix has an unexpected row count.")

# The model learns few components with many zero loadings.
model = MiniBatchSparsePCA(
    n_components=3,
    alpha=1.0,
    batch_size=3,
    random_state=42,
)

# Fit once, then transform documents into compact coordinates.
document_scores = model.fit_transform(word_counts)
component_weights = model.components_

# Show the strongest words for each sparse component.
print("scikit-learn version:", sklearn.__version__)
print("Original matrix shape:", word_counts.shape)
print("Reduced matrix shape:", document_scores.shape)

# Count near-zero weights to show sparsity directly.
zero_share = np.mean(np.abs(component_weights) < 1e-9)
print("Share of zero component weights:", round(float(zero_share), 2))

# Display only the top words for beginner-friendly interpretation.
for component_index in range(component_weights.shape[0]):
    weights = component_weights[component_index]
    top_indices = np.argsort(np.abs(weights))[-3:][::-1]
    top_words = ", ".join(feature_names[top_indices])
    print("Component", component_index + 1, "top words:", top_words)

# Show one transformed document as compact numeric features.
rounded_scores = np.round(document_scores[0], 2)
print("First document compact features:", rounded_scores)



### **3.2. Kernel PCA**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_A/image_03_02.jpg?v=1784025068" width="250">



>* Extends PCA to nonlinear data patterns
>* Uses kernels to reveal hidden structure

>* TruncatedSVD efficiently reduces sparse text matrices
>* Kernel PCA suits small nonlinear document sets

>* Use TruncatedSVD for large sparse text data
>* Use Kernel PCA for small nonlinear exploration



In [ ]:
#@title Python Code - Kernel PCA

# Compare nonlinear Kernel PCA with TruncatedSVD.
# Use small sparse text-like document features.
# Visualize when nonlinear structure may appear.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import KernelPCA
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
import sklearn

# These short documents form three simple topic groups.
documents = [
    "court judge legal contract evidence",
    "lawyer court evidence trial contract",
    "judge trial legal court lawyer",
    "doctor patient medicine treatment symptom",
    "hospital doctor treatment patient medicine",
    "symptom diagnosis patient hospital doctor",
    "software code computer data algorithm",
    "computer program data code software",
    "algorithm data program computer code",
]

# TF-IDF creates a sparse document-term matrix.
vectorizer = TfidfVectorizer()
sparse_text_matrix = vectorizer.fit_transform(documents)

# Validate the small sparse matrix before decomposition.
if sparse_text_matrix.shape[0] != len(documents):
    raise ValueError("Each document should become one matrix row.")

# TruncatedSVD works directly on sparse text features.
svd = TruncatedSVD(n_components=2, random_state=42)
svd_points = svd.fit_transform(sparse_text_matrix)

# Kernel PCA needs pairwise nonlinear similarities.
kernel_pca = KernelPCA(n_components=2, kernel="rbf", gamma=0.8)
kpca_points = kernel_pca.fit_transform(sparse_text_matrix)

# Print concise facts that connect method choice to text data.
print("scikit-learn version:", sklearn.__version__)
print("Sparse text matrix shape:", sparse_text_matrix.shape)
print("TruncatedSVD explained variance:", np.round(svd.explained_variance_ratio_, 3))
print("Kernel PCA output shape:", kpca_points.shape)

# Plot Kernel PCA coordinates for the tiny document collection.
topic_labels = np.array(["legal", "medical", "technology"])
topic_index = np.repeat([0, 1, 2], 3)
colors = np.array(["tab:blue", "tab:green", "tab:orange"])

fig, ax = plt.subplots(figsize=(7, 5))
for topic_number, topic_name in enumerate(topic_labels):
    selected = topic_index == topic_number
    ax.scatter(
        kpca_points[selected, 0],
        kpca_points[selected, 1],
        label=topic_name,
        color=colors[topic_number],
        s=80,
    )

ax.set_title("Kernel PCA on a tiny sparse text example")
ax.set_xlabel("Kernel PCA component 1")
ax.set_ylabel("Kernel PCA component 2")
ax.legend()
plt.show()



### **3.3. Sparse Text Reduction**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_24/Lecture_A/image_03_03.jpg?v=1784025070" width="250">



>* Text data creates huge sparse feature spaces
>* Truncated SVD finds compact latent themes

>* Reveals shared meaning beyond exact words
>* Improves text clustering, search, and classification

>* Balance component count for clarity and performance
>* Validate results while preserving sparse text efficiency



In [ ]:
#@title Python Code - Sparse Text Reduction

# Reduce sparse text into latent semantic features.
# TruncatedSVD works without densifying text matrices.
# The output compares original and reduced dimensions.

import numpy as np
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# These short documents form three simple topic groups.
documents = [
    "doctor hospital treatment patient medicine clinic",
    "physician clinic therapy patient health hospital",
    "team scored goal match coach league season",
    "player coach tournament goal sports team",
    "software computer data algorithm network system",
    "technology software hardware data computer network",
]

# TF-IDF creates a sparse document by term matrix.
vectorizer = TfidfVectorizer()
sparse_text_matrix = vectorizer.fit_transform(documents)

# Validate that the example produced a sparse text matrix.
if sparse_text_matrix.shape[0] != len(documents):
    raise ValueError("The document count does not match the matrix rows.")

# TruncatedSVD reduces sparse text into two latent features.
svd = TruncatedSVD(n_components=2, random_state=42)
reduced_text_matrix = svd.fit_transform(sparse_text_matrix)

# Components can be inspected through their strongest terms.
terms = np.array(vectorizer.get_feature_names_out())
first_component_order = np.argsort(np.abs(svd.components_[0]))[::-1]
top_terms = terms[first_component_order[:5]]

print("scikit-learn version:", sklearn.__version__)
print("Original sparse shape:", sparse_text_matrix.shape)
print("Reduced dense shape:", reduced_text_matrix.shape)
print("Explained variance ratio:", np.round(svd.explained_variance_ratio_, 3))
print("Top terms in component 1:", ", ".join(top_terms))
print("First document reduced values:", np.round(reduced_text_matrix[0], 3))



# <font color="#418FDE" size="6.5" uppercase>**PCA SVD**</font>


In this lecture, you learned to:
- Apply PCA and interpret explained variance and components. 
- Use incremental, sparse, kernel, and mini-batch PCA variants appropriately. 
- Apply TruncatedSVD for sparse high-dimensional data such as text features. 

In the next Lecture (Lecture B), we will go over 'Factor Components'